# Drift Evaluation — Episodios Abruptos
 - Carga de datos sintéticos (`synthetic_plant.csv`) y etiquetado manual (`synthetic_plant_events.csv`)
 - Se filtran SOLO episodios manuales con drift_type = 'abrupt'
 - Detección de drift (psi, ks, wasserstein), construcción de episodios automáticos
 - Evaluación vs manual (F1 por episodios + F1_time, delay, false alarms)
 - Export de CSVs y plots (Matplotlib)


In [ ]:
# %% [markdown]
# # Drift Evaluation — Episodios Abruptos
#
# - Carga de datos sintéticos (`synthetic_plant.csv`) y etiquetado manual (`synthetic_plant_events.csv`)
# - Se filtran SOLO episodios manuales con drift_type = 'abrupt'
# - Detección de drift (psi, ks, wasserstein), construcción de episodios automáticos
# - Evaluación vs manual (F1 por episodios + F1_time, delay, false alarms)
# - Export de CSVs y plots (Matplotlib)

# %% 1. Imports
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import importlib.util
import plotly.express as px

# %% 2. Cargar módulo Funciones_Drift.py y alias útiles
spec = importlib.util.spec_from_file_location("funciones_drift", "../Analisis/Funciones_Drift.py")
funciones_drift = importlib.util.module_from_spec(spec)
spec.loader.exec_module(funciones_drift)

strip_outliers         = funciones_drift.strip_outliers
ref_decay_prefix_mass  = funciones_drift.ref_decay_prefix_mass
ref_golden             = funciones_drift.ref_golden
ref_seasonal           = funciones_drift.ref_seasonal
score_numeric_series   = funciones_drift._score_numeric_series

# %% 3. Carga de serie sintética y construcción de intervalos manuales

SERIES_PATH = Path("synthetic_data/synthetic_plant.csv")
LABELS_PATH = Path("synthetic_data/synthetic_plant_events.csv")

assert SERIES_PATH.exists(), f"No se encontró {SERIES_PATH}"
assert LABELS_PATH.exists(),  f"No se encontró {LABELS_PATH}"

# --- Serie sintética ---
df_raw = pd.read_csv(SERIES_PATH)

if "date_time" not in df_raw.columns:
    for c in ["datetime", "timestamp", "time", "fecha", "tiempo"]:
        if c in df_raw.columns:
            df_raw = df_raw.rename(columns={c: "date_time"})
            break

df_raw["date_time"] = pd.to_datetime(df_raw["date_time"], errors="coerce")
df_raw = (
    df_raw
    .dropna(subset=["date_time"])
    .sort_values("date_time")
    .set_index("date_time")
)

df_raw = strip_outliers(df_raw)
df = df_raw.select_dtypes(include="number").copy()
assert not df.empty, "No hay columnas numéricas en la serie sintética."

t_min, t_max = df.index.min(), df.index.max()

# --- Etiquetado manual ---
events = pd.read_csv(LABELS_PATH)
events["date_time"] = pd.to_datetime(events["date_time"], errors="coerce")
events = (
    events
    .dropna(subset=["date_time", "variable", "event"])
    .assign(event=lambda s: s["event"].str.lower().str.strip())
    .query("event in ['start','end']")
    .sort_values(["variable", "date_time"])
    .reset_index(drop=True)
)

def events_to_intervals(ev: pd.DataFrame) -> pd.DataFrame:
    """
    Convierte pares start/end por variable en episodios continuos (manuales).
    Si existe columna 'drift_type' en el CSV de eventos, la conserva por episodio.
    Se asume que start y end de un mismo episodio tienen el mismo drift_type.
    """
    has_type = "drift_type" in ev.columns
    rows = []
    for var, g in ev.groupby("variable", sort=True):
        open_t = None
        open_type = None
        for _, r in g.iterrows():
            evt = str(r["event"]).lower()
            dt_val = r["drift_type"] if has_type else "unknown"

            if evt == "start":
                open_t = r["date_time"]
                open_type = dt_val
            elif evt == "end" and open_t is not None and r["date_time"] > open_t:
                rows.append({
                    "variable": var,
                    "manual_start": open_t,
                    "manual_end": r["date_time"],
                    "drift_type": open_type,
                })
                open_t = None
                open_type = None
    return pd.DataFrame(rows)

intervals_manual_all = events_to_intervals(events)


In [ ]:
intervals_manual = (
    intervals_manual_all[
        intervals_manual_all["drift_type"].astype(str).str.lower() == "gradual"
    ]
    .copy()
)
if intervals_manual.empty:
    raise ValueError("No hay episodios manuales con drift_type = 'gradual'.")
intervals_manual = intervals_manual.sort_values(
    ["variable", "manual_start"]
).reset_index(drop=True)
print(f"Episodios manuales GRADUALES: {len(intervals_manual)}")

In [ ]:
EVAL_WINDOWS = ["6H", "12H", "24H", "36H", "48H"]
STRATEGIES   = ["decay", "golden", "seasonal"]
METRICS      = ("psi", "ks", "wasserstein")   # por ahora todas

OUTPUT_DIR = Path("synthetic_data/results_gradual")